# Stage 3 — Skill Mapping and Career Transition Recommender

## 3A. Build role-level skill profiles

The supplied dataset contains one `Required_Skills` category per job-market
observation rather than a complete skill inventory for each occupation.
Therefore, skill profiles are constructed by aggregating observations within
each `Job_Title`.

For each occupation, the profile represents the proportion of its observations
associated with each recorded skill category. These profiles provide a
dataset-based representation for comparing occupations and demonstrating
skill adjacency.

Because the supplied role-skill relationships are coarse and sometimes
counterintuitive, the resulting similarities are treated as decision-support
signals rather than validated real-world career pathways.

In [1]:
import pandas as pd
import numpy as np

# Load the original hackathon dataset
df = pd.read_csv("../data/raw/ai_job_market_insights.csv")

print("Dataset shape:", df.shape)

print(
    df[
        [
            "Job_Title",
            "Required_Skills",
            "Job_Growth_Projection",
            "Automation_Risk",
            "Salary_USD",
        ]
    ]
    .head()
    .to_string(index=False)
)

Dataset shape: (500, 10)
            Job_Title    Required_Skills Job_Growth_Projection Automation_Risk    Salary_USD
Cybersecurity Analyst       UX/UI Design                Growth            High 111392.165243
 Marketing Specialist          Marketing               Decline            High  93792.562466
        AI Researcher       UX/UI Design                Growth            High 107170.263069
        Sales Manager Project Management                Growth            High  93027.953758
Cybersecurity Analyst         JavaScript               Decline             Low  87752.922171


In [2]:
print("Number of job titles:", df["Job_Title"].nunique())
print("Number of skill categories:", df["Required_Skills"].nunique())

print("\nJob titles:")
print(sorted(df["Job_Title"].unique()))

print("\nSkill categories:")
print(sorted(df["Required_Skills"].unique()))

Number of job titles: 10
Number of skill categories: 10

Job titles:
['AI Researcher', 'Cybersecurity Analyst', 'Data Scientist', 'HR Manager', 'Marketing Specialist', 'Operations Manager', 'Product Manager', 'Sales Manager', 'Software Engineer', 'UX Designer']

Skill categories:
['Communication', 'Cybersecurity', 'Data Analysis', 'JavaScript', 'Machine Learning', 'Marketing', 'Project Management', 'Python', 'Sales', 'UX/UI Design']


In [3]:
# Build a role-by-skill profile
# Each value is the proportion of observations for a role
# associated with each Required_Skills category.

role_skill_profile = pd.crosstab(
    df["Job_Title"], df["Required_Skills"], normalize="index"
)

print(role_skill_profile.round(3).to_string())

Required_Skills        Communication  Cybersecurity  Data Analysis  JavaScript  Machine Learning  Marketing  Project Management  Python  Sales  UX/UI Design
Job_Title                                                                                                                                                   
AI Researcher                  0.078          0.098          0.078       0.176             0.039      0.078               0.059   0.176  0.137         0.078
Cybersecurity Analyst          0.055          0.182          0.055       0.073             0.127      0.091               0.091   0.091  0.073         0.164
Data Scientist                 0.032          0.145          0.113       0.048             0.097      0.081               0.113   0.145  0.129         0.097
HR Manager                     0.070          0.105          0.123       0.088             0.175      0.088               0.105   0.053  0.105         0.088
Marketing Specialist           0.125          0.062       

In [4]:
# Validate that each role's skill proportions sum to 1 (100%)

profile_check = role_skill_profile.sum(axis=1)

print("Minimum row sum:", profile_check.min())
print("Maximum row sum:", profile_check.max())

assert np.allclose(profile_check.values, 1.0)

print("\n✓ All role skill profiles sum to 1.")
print("Role-skill matrix shape:", role_skill_profile.shape)

Minimum row sum: 0.9999999999999999
Maximum row sum: 1.0

✓ All role skill profiles sum to 1.
Role-skill matrix shape: (10, 10)


## 3B. Calculate similarity between occupations

Cosine similarity is used to compare the skill-profile vectors created in
Section 3A. It measures how similar the distribution of recorded skill
categories is between two occupations.

A higher cosine similarity indicates greater overlap in the skill patterns
observed in the supplied dataset. Because these profiles are derived from one
recorded skill category per observation, similarity scores are interpreted as
dataset-based skill-adjacency signals rather than validated measures of
real-world occupational similarity.

In [5]:
from sklearn.metrics.pairwise import cosine_similarity

# Calculate pairwise cosine similarity between role skill profiles
similarity_matrix = cosine_similarity(role_skill_profile)

role_similarity = pd.DataFrame(
    similarity_matrix, index=role_skill_profile.index, columns=role_skill_profile.index
)

print(role_similarity.round(3).to_string())

Job_Title              AI Researcher  Cybersecurity Analyst  Data Scientist  HR Manager  Marketing Specialist  Operations Manager  Product Manager  Sales Manager  Software Engineer  UX Designer
Job_Title                                                                                                                                                                                        
AI Researcher                  1.000                  0.804           0.874       0.796                 0.914               0.905            0.806          0.751              0.802        0.729
Cybersecurity Analyst          0.804                  1.000           0.922       0.904                 0.747               0.915            0.874          0.855              0.860        0.798
Data Scientist                 0.874                  0.922           1.000       0.910                 0.837               0.910            0.851          0.944              0.888        0.850
HR Manager                    

In [6]:
# Validate the similarity matrix

assert role_similarity.shape == (10, 10)
assert np.allclose(np.diag(role_similarity), 1.0)
assert np.allclose(role_similarity.values, role_similarity.values.T)

print("✓ Similarity matrix is 10 × 10.")
print("✓ Each role has self-similarity of 1.")
print("✓ Similarity matrix is symmetric.")

✓ Similarity matrix is 10 × 10.
✓ Each role has self-similarity of 1.
✓ Similarity matrix is symmetric.


## 3C. Generate skill-adjacent candidate occupations

For a selected current occupation, all other occupations are ranked by cosine
similarity to its observed skill profile. The current occupation itself is
excluded.

Because similarity scores are generally high across this synthetic dataset,
no absolute similarity threshold is imposed. Instead, cosine similarity is
used as a relative ranking signal to identify the occupations whose observed
skill profiles are most similar to the selected role.

In [7]:
def get_skill_candidates(current_role, top_n=5):
    """Return the most skill-similar alternative roles."""

    if current_role not in role_similarity.index:
        raise ValueError(f"Unknown role: {current_role}")

    candidates = (
        role_similarity.loc[current_role]
        .drop(current_role)
        .sort_values(ascending=False)
        .head(top_n)
        .rename("skill_similarity")
        .reset_index()
        .rename(columns={"Job_Title": "candidate_role"})
    )

    return candidates

In [8]:
test_role = "Data Scientist"

skill_candidates = get_skill_candidates(test_role, top_n=5)

print(f"Top skill-adjacent roles for: {test_role}\n")
print(skill_candidates.round(3).to_string(index=False))

Top skill-adjacent roles for: Data Scientist

       candidate_role  skill_similarity
        Sales Manager             0.944
Cybersecurity Analyst             0.922
   Operations Manager             0.910
           HR Manager             0.910
    Software Engineer             0.888


## 3D. Add role-level market signals

Skill similarity alone is insufficient for evaluating a potential career
transition. Candidate occupations are therefore supplemented with three
descriptive signals available in the supplied dataset:

- **Growth share:** proportion of observations for the occupation labelled
  `Growth` in `Job_Growth_Projection`.
- **High-risk share:** proportion of observations for the occupation labelled
  `High` in `Automation_Risk`.
- **Median salary:** median `Salary_USD` observed for the occupation.

These are descriptive summaries of the supplied dataset, not forecasts of
future labour-market outcomes. Observed automation-risk labels are used rather
than predictions from the Stage 2 classifier because held-out validation did
not demonstrate reliable predictive performance.

In [9]:
# Build descriptive market signals for each occupation

role_summary = (
    df.groupby("Job_Title")
    .agg(
        postings=("Job_Title", "size"),
        median_salary=("Salary_USD", "median"),
        growth_share=("Job_Growth_Projection", lambda x: (x == "Growth").mean()),
        decline_share=("Job_Growth_Projection", lambda x: (x == "Decline").mean()),
        high_risk_share=("Automation_Risk", lambda x: (x == "High").mean()),
    )
    .reset_index()
)

print(role_summary.round(3).to_string(index=False))

            Job_Title  postings  median_salary  growth_share  decline_share  high_risk_share
        AI Researcher        51      98036.517         0.392          0.314            0.392
Cybersecurity Analyst        55      88642.445         0.255          0.345            0.418
       Data Scientist        62      91674.523         0.355          0.371            0.274
           HR Manager        57      92250.314         0.246          0.386            0.263
 Marketing Specialist        48      92719.580         0.229          0.479            0.354
   Operations Manager        44      97466.493         0.432          0.159            0.227
      Product Manager        39      91339.636         0.385          0.256            0.410
        Sales Manager        49      93002.693         0.388          0.327            0.408
    Software Engineer        41      84124.067         0.390          0.293            0.366
          UX Designer        54      89556.393         0.352          

In [10]:
# Validate role-level summaries

assert role_summary["Job_Title"].nunique() == df["Job_Title"].nunique()
assert role_summary["postings"].sum() == len(df)

for col in ["growth_share", "decline_share", "high_risk_share"]:
    assert role_summary[col].between(0, 1).all()

print("✓ One summary row created for each occupation.")
print("✓ Role observation counts sum to the full dataset.")
print("✓ Growth, decline and high-risk shares are bounded between 0 and 1.")

✓ One summary row created for each occupation.
✓ Role observation counts sum to the full dataset.
✓ Growth, decline and high-risk shares are bounded between 0 and 1.


## 3E. Score and rank potential career transitions

Candidate occupations are ranked using a transparent transition score that
combines skill adjacency with the descriptive growth and automation-risk
signals available in the supplied dataset:

**Transition Score = 0.60 × Skill Similarity  
+ 0.25 × Growth Share  
+ 0.15 × (1 − High-Risk Share)**

Skill similarity receives the greatest weight because the primary objective is
to identify transitions that make use of the worker's existing skill profile.
Growth and automation risk provide secondary directional context.

The weights are heuristic design choices for this prototype; they are not
estimated from data and should not be interpreted as validated estimates of
career-transition success.

In [11]:
def rank_transitions(current_role, top_n=3):
    """Rank alternative occupations using skill and market signals."""

    # Start with all alternative roles
    candidates = get_skill_candidates(current_role, top_n=len(role_similarity) - 1)

    # Add descriptive market signals
    candidates = candidates.merge(
        role_summary, left_on="candidate_role", right_on="Job_Title", how="left"
    ).drop(columns="Job_Title")

    # Transparent heuristic transition score
    candidates["transition_score"] = (
        0.60 * candidates["skill_similarity"]
        + 0.25 * candidates["growth_share"]
        + 0.15 * (1 - candidates["high_risk_share"])
    )

    # Rank and return the strongest candidates
    candidates = (
        candidates.sort_values("transition_score", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )

    return candidates

In [12]:
test_transitions = rank_transitions("Data Scientist", top_n=3)

display_cols = [
    "candidate_role",
    "skill_similarity",
    "growth_share",
    "high_risk_share",
    "median_salary",
    "transition_score",
]

print(test_transitions[display_cols].round(3).to_string(index=False))

    candidate_role  skill_similarity  growth_share  high_risk_share  median_salary  transition_score
Operations Manager             0.910         0.432            0.227      97466.493             0.770
     Sales Manager             0.944         0.388            0.408      93002.693             0.752
 Software Engineer             0.888         0.390            0.366      84124.067             0.726


## 3F. Explain shared skills and reskilling gaps

To make each suggested transition interpretable, the recommender compares the
current and destination role-level skill profiles.

Because all occupations contain observations across most skill categories,
simple presence/absence overlap would provide little information. Instead:

- **Shared strengths** are skill categories with relatively strong representation
  across both the current and destination profiles.
- **Skills to strengthen** are categories more strongly represented in the
  destination occupation than in the current occupation.

These outputs describe differences in the supplied dataset's aggregated skill
profiles and should be interpreted as reskilling signals rather than complete
individual skill-gap assessments.

In [13]:
def explain_transition(current_role, candidate_role, top_n=3):
    """Identify shared strengths and destination-oriented skill gaps."""

    current = role_skill_profile.loc[current_role]
    candidate = role_skill_profile.loc[candidate_role]

    comparison = pd.DataFrame(
        {"current_profile": current, "candidate_profile": candidate}
    )

    # Skills strongly represented across both profiles
    comparison["shared_strength"] = (
        comparison["current_profile"] + comparison["candidate_profile"]
    ) / 2

    # Positive values indicate greater representation
    # in the destination occupation
    comparison["skill_gap"] = (
        comparison["candidate_profile"] - comparison["current_profile"]
    )

    shared_strengths = (
        comparison.sort_values("shared_strength", ascending=False)
        .head(top_n)
        .index.tolist()
    )

    skills_to_strengthen = (
        comparison[comparison["skill_gap"] > 0]
        .sort_values("skill_gap", ascending=False)
        .head(top_n)
        .index.tolist()
    )

    return shared_strengths, skills_to_strengthen

In [14]:
shared, strengthen = explain_transition("Data Scientist", "Operations Manager")

print("Transition: Data Scientist → Operations Manager")
print("\nShared strengths:")
for skill in shared:
    print(f"  • {skill}")

print("\nSkills to strengthen:")
for skill in strengthen:
    print(f"  • {skill}")

Transition: Data Scientist → Operations Manager

Shared strengths:
  • Cybersecurity
  • Python
  • Sales

Skills to strengthen:
  • Communication
  • JavaScript
  • UX/UI Design


## 3G. Assemble the career transition recommender

The final recommender combines the ranked transition candidates with
interpretable skill information. For a selected occupation, it returns the
top three dataset-supported transition candidates together with skill
similarity, growth and automation-risk context, median salary, shared
strengths, skills to strengthen, and the transparent transition score.

The output is intended as an exploratory decision-support prototype rather
than a validated career recommendation system.

In [15]:
def recommend_careers(current_role, top_n=3):
    """Return ranked career transitions with skill explanations."""

    recommendations = rank_transitions(current_role, top_n=top_n).copy()

    shared_list = []
    strengthen_list = []

    for candidate_role in recommendations["candidate_role"]:
        shared, strengthen = explain_transition(current_role, candidate_role)

        shared_list.append(", ".join(shared))
        strengthen_list.append(", ".join(strengthen))

    recommendations["shared_strengths"] = shared_list
    recommendations["skills_to_strengthen"] = strengthen_list

    return recommendations

In [16]:
final_recommendations = recommend_careers("Data Scientist", top_n=3)

final_cols = [
    "candidate_role",
    "skill_similarity",
    "growth_share",
    "high_risk_share",
    "median_salary",
    "shared_strengths",
    "skills_to_strengthen",
    "transition_score",
]

print(final_recommendations[final_cols].round(3).to_string(index=False))

    candidate_role  skill_similarity  growth_share  high_risk_share  median_salary                                    shared_strengths                                skills_to_strengthen  transition_score
Operations Manager             0.910         0.432            0.227      97466.493                        Cybersecurity, Python, Sales             Communication, JavaScript, UX/UI Design             0.770
     Sales Manager             0.944         0.388            0.408      93002.693                Cybersecurity, Data Analysis, Python Data Analysis, Machine Learning, Project Management             0.752
 Software Engineer             0.888         0.390            0.366      84124.067 Machine Learning, Data Analysis, Project Management          Communication, Machine Learning, Marketing             0.726


## 3H. Validation, interpretation and limitations

The career-transition recommender successfully generates ranked alternative
occupations for each role represented in the supplied dataset. Recommendations
combine three transparent signals: observed skill-profile similarity (60%),
growth share (25%), and inverse high-risk share (15%). The resulting score is
a heuristic ranking mechanism rather than a trained or validated prediction
of career-transition success.

For the demonstration case of `Data Scientist`, the highest-ranked transition
is `Operations Manager`, followed by `Sales Manager` and `Software Engineer`.
The ranking illustrates how the recommender balances skill adjacency with
descriptive growth and automation-risk signals rather than relying on skill
similarity alone.

Several limitations constrain interpretation:

1. `Required_Skills` contains only one skill category per observation rather
   than a complete occupational skill inventory.
2. Role-skill profiles are unusually diffuse, producing generally high cosine
   similarities across occupations.
3. Some observed role-skill relationships are counterintuitive and should not
   be interpreted as validated real-world occupational relationships.
4. Growth and automation-risk values are categorical labels in the supplied
   dataset rather than longitudinal labour-market forecasts.
5. The 60/25/15 weighting scheme is a transparent prototype design choice and
   has not been empirically validated.
6. Shared strengths and skills-to-strengthen are inferred from aggregated
   role-level profiles rather than an individual worker's actual skills.
7. External occupational data such as O*NET could substantially improve skill
   granularity and real-world validity in future development.

Accordingly, the recommender should be interpreted as a transparent
proof-of-concept for skill-adjacent career exploration, not as a validated
career-advice or labour-market forecasting system.

In [17]:
# Final functional check: confirm the recommender works for every role

for role in role_skill_profile.index:
    result = recommend_careers(role, top_n=3)

    assert len(result) == 3
    assert role not in result["candidate_role"].values
    assert result["transition_score"].between(0, 1).all()

print(
    f"✓ Recommender successfully tested for all {len(role_skill_profile)} occupations."
)
print("✓ Each role returns 3 alternative occupations.")
print("✓ Current occupation is excluded from its own recommendations.")
print("✓ All transition scores are bounded between 0 and 1.")

✓ Recommender successfully tested for all 10 occupations.
✓ Each role returns 3 alternative occupations.
✓ Current occupation is excluded from its own recommendations.
✓ All transition scores are bounded between 0 and 1.


## 3I. Preserve recommender artifacts

The final role-level skill profiles and descriptive market summaries are exported
for use by the Streamlit application.

The application will reconstruct career-transition recommendations from these
preserved Stage 3 outputs using the same transparent scoring logic developed in
this notebook.

This keeps the interface separate from the analytical workflow while avoiding
the need to rerun the Stage 3 notebook when the application starts.

In [ ]:
from pathlib import Path

APP_DATA_DIR = Path("../data/processed")
APP_DATA_DIR.mkdir(parents=True, exist_ok=True)

role_skill_path = APP_DATA_DIR / "role_skill_profiles.csv"
role_summary_path = APP_DATA_DIR / "role_market_summary.csv"

role_skill_profile.to_csv(role_skill_path)
role_summary.to_csv(role_summary_path, index=False)

print(f"Saved: {role_skill_path}")
print(f"Saved: {role_summary_path}")

print("\nArtifact check:")
for path in [role_skill_path, role_summary_path]:
    print(f"{path}: {'OK' if path.exists() else 'MISSING'}")

Saved: ..\data\processed\role_skill_profiles.csv
Saved: ..\data\processed\role_market_summary.csv

Artifact check:
..\data\processed\role_skill_profiles.csv: OK
..\data\processed\role_market_summary.csv: OK


: 